# 🏠 End-to-End House Price Predictor
## 🧹 Phase 2: Data Cleaning

**Goal**: Transform raw, messy data into a clean, model-ready dataset.

### What we do in this phase:
1. **Remove outliers** — extreme homes that distort regression boundaries
2. **Log-transform `SalePrice`** — stabilise variance, normalise the target
3. **Impute missing values** — domain-aware strategies (not just fill with mean)
4. **Fix rare categories** — group infrequent labels into `"Other"`
5. **Save cleaned output** — to `data/processed/` for use in Phase 3

> All reusable logic lives in `src/cleaning.py` so future phases can import it cleanly.


---
### 1. Imports & Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

# Allow importing from src/
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from src.cleaning import (
    remove_outliers,
    log_transform_target,
    impute_missing,
    fix_rare_categories,
    fix_dtypes,
    cleaning_report,
)

DATA_DIR      = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#1a1d27",
    "axes.edgecolor": "#3a3d4d", "axes.labelcolor": "#c9d1d9",
    "xtick.color": "#8b949e", "ytick.color": "#8b949e",
    "text.color": "#c9d1d9", "grid.color": "#21262d",
    "grid.linewidth": 0.6, "font.size": 11,
})
ACCENT, ACCENT2, ACCENT3 = "#58a6ff", "#3fb950", "#ff7b72"
print("✔ Setup complete")


---
### 2. Load Raw Data


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
print(f"Train : {train.shape}   Test : {test.shape}")
train.head(3)


---
### 3. Remove Outliers

The dataset author (Dean De Cock) explicitly flagged two homes:
- `GrLivArea > 4000 sq ft` AND `SalePrice < $200,000`
- These are partial/agricultural sales — statistically extreme and misleading for a regression model.

Removing them **before** the log-transform prevents their leverage from skewing learned coefficients.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("GrLivArea vs SalePrice — Before & After Outlier Removal", fontsize=13, color="#e6edf3", y=1.01)

# Before
axes[0].scatter(train["GrLivArea"], train["SalePrice"], alpha=0.4, s=18, color=ACCENT, edgecolors="none")
suspects = train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 200_000)]
axes[0].scatter(suspects["GrLivArea"], suspects["SalePrice"], color=ACCENT3, s=60, zorder=5, label=f"Outliers ({len(suspects)})")
axes[0].set_title("Before", color="#e6edf3"); axes[0].legend(); axes[0].yaxis.grid(True, alpha=0.3)

train_clean = remove_outliers(train)

# After
axes[1].scatter(train_clean["GrLivArea"], train_clean["SalePrice"], alpha=0.4, s=18, color=ACCENT2, edgecolors="none")
axes[1].set_title(f"After  (removed {len(train) - len(train_clean)} rows)", color="#e6edf3"); axes[1].yaxis.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Rows removed: {len(train) - len(train_clean)}  |  Remaining: {len(train_clean)}")


---
### 4. Log-Transform `SalePrice`

Linear models assume normally distributed errors. A right-skewed target (skew = 1.88) violates this.

`np.log1p(x)` = `log(1 + x)` — safe even if x=0, and easily reversed with `np.expm1` at prediction time.


In [ ]:
train_clean = log_transform_target(train_clean)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Still has original SalePrice in test; show train target distribution
orig_sp = pd.read_csv(DATA_DIR / "train.csv")["SalePrice"]
axes[0].hist(orig_sp, bins=50, color=ACCENT3, edgecolor="none", alpha=0.85)
axes[0].set_title(f"Original  (skew={orig_sp.skew():.2f})", color="#e6edf3"); axes[0].set_xlabel("SalePrice")
axes[1].hist(train_clean["SalePrice"], bins=50, color=ACCENT2, edgecolor="none", alpha=0.85)
axes[1].set_title(f"log1p  (skew={train_clean['SalePrice'].skew():.2f})", color="#e6edf3"); axes[1].set_xlabel("log1p(SalePrice)")
for ax in axes: ax.yaxis.grid(True, alpha=0.3)
plt.suptitle("SalePrice: Before vs After Log Transform", fontsize=13, color="#e6edf3", y=1.01)
plt.tight_layout(); plt.show()


---
### 5. Missing Value Imputation

This is the most important step in cleaning. Wrong imputation = silent data corruption.

**Strategy** (from `data/raw/data_description.txt`):

| Strategy | Columns | Reason |
|---|---|---|
| Fill `"None"` | `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`, `GarageType/Finish/Qual/Cond`, `BsmtQual/Cond/Exposure/FinType*`, `MasVnrType` | `NaN` literally means "no such feature exists" |
| Fill `0` | `GarageYrBlt`, `GarageCars`, `GarageArea`, `MasVnrArea`, `BsmtFinSF1/2`, `BsmtUnfSF`, `TotalBsmtSF`, `BsmtFullBath`, `BsmtHalfBath` | Numeric counterparts to the None-filled columns |
| Fill **median by Neighborhood** | `LotFrontage` | Street frontage strongly varies by neighborhood; group median is most accurate |
| Fill **mode** | `MSZoning`, `Electrical`, `KitchenQual`, `Exterior1st`, `Exterior2nd`, `SaleType`, `Functional` | Single or very few missing — mode is safe |


In [ ]:
missing_before = train_clean.isnull().sum().sum()
train_clean, test_clean = impute_missing(train_clean, test)
missing_after_train = train_clean.isnull().sum().sum()
missing_after_test  = test_clean.isnull().sum().sum()

print(f"Train — missing before: {missing_before:,}  →  after: {missing_after_train}")
print(f"Test  — missing after : {missing_after_test}")

# Visualise remaining (should be 0)
remaining = train_clean.isnull().sum()
remaining = remaining[remaining > 0]
if remaining.empty:
    print("\n✅ No missing values remain in train set.")
else:
    print("\n⚠️  Still missing:\n", remaining)


---
### 6. Fix Rare Categories

Categories that appear in < 1% of rows cause problems:
- They create very sparse One-Hot columns
- They may appear in train but not test (or vice versa), causing shape mismatches

We replace them with `"Other"`.


In [ ]:
cat_cols = train_clean.select_dtypes(include="object").columns.tolist()
train_clean, test_clean = fix_rare_categories(train_clean, test_clean, cat_cols, threshold=0.01)
print("Rare categories consolidated into 'Other'.")

# Quick check — show category counts for a high-cardinality column
print("\nNeighborhood value counts (post-cleaning):")
print(train_clean["Neighborhood"].value_counts().to_string())


---
### 7. Fix Data Types

Some numerical-looking columns are actually ordinal **codes**, not continuous measurements.
E.g. `MSSubClass` (20, 30, 40…) encodes dwelling types — treating as integer implies a false ordering.
We cast these to `str` so they get One-Hot encoded in Phase 3.


In [ ]:
train_clean = fix_dtypes(train_clean)
test_clean  = fix_dtypes(test_clean)
print("MSSubClass dtype:", train_clean["MSSubClass"].dtype)
print("YrSold dtype    :", train_clean["YrSold"].dtype)


---
### 8. Cleaning Summary Report


In [ ]:
cleaning_report(train_clean, test_clean)


---
### 9. Save Cleaned Data

Saving to `data/processed/` — these files are gitignored (too large, reproducible from this notebook).
Phase 3 will read from here.


In [ ]:
train_clean.to_csv(PROCESSED_DIR / "train_cleaned.csv", index=False)
test_clean.to_csv(PROCESSED_DIR  / "test_cleaned.csv",  index=False)

print(f"✅ Saved train_cleaned.csv  — {train_clean.shape}")
print(f"✅ Saved test_cleaned.csv   — {test_clean.shape}")


---
## ✅ Phase 2 Complete

| Step | Result |
|---|---|
| Outliers removed | 2 rows |
| SalePrice | log1p transformed (skew: 1.88 → ~0.12) |
| Missing values | All imputed (domain-aware strategies) |
| Rare categories | Consolidated into `"Other"` |
| Dtype fixes | `MSSubClass`, `YrSold`, `MoSold` → string |
| Output | `data/processed/train_cleaned.csv` & `test_cleaned.csv` |

**Next → Phase 3: Feature Engineering** — create new columns, encode categoricals, fix skewed features.
